# Searching for Koopman eigenfunctions

We search for polynomial Koopman eigenfunctions of the van der Pol system. A polynomial observable $P$ is an eigenfunction with eigenvalue $\lambda$ when

$$\frac{dP}{dt} - \lambda P = 0.$$

The polynomial coefficients of this expression provide algebraic equations for the unknown coefficients of $P$ and $\lambda$.

In [11]:
import sympy as sy
from IPython.display import Math, display

from symode.componentwise_expression_factory import (
    create_componentwise_expression_with_monomial_bases_from_polynomial_expression,
    create_parametrized_polynomial,
)
from symode.dynamical_system import DynamicalSystem
from symode.root_finder import get_reduced_expression

## The dynamical system

In [12]:
system = DynamicalSystem("lorenz")
Math(str(system))

<IPython.core.display.Math object>

## Polynomial ansatz

Set `max_degree` to choose the maximum total degree of the polynomial. For every exponent tuple, the coefficient is named using that tuple: for example, `a_0_0_0` multiplies the constant term and `a_1_0_0` multiplies the first system variable.

In [13]:
max_degree = 2
polynomial = create_parametrized_polynomial(max_degree, system.get_variables())

## Time derivative and eigenfunction remainder

The system computes the time derivative using its vector field. We then subtract $\lambda P$ and expand the result as a polynomial in the state variables.

In [14]:
lambda_ = sy.Symbol("lambda")
remainder = sy.expand(system.get_operator_application(polynomial.sum_up(), lambda_))
remainder

-a_0_0_0*lambda - a_0_0_1*beta*z - a_0_0_1*lambda*z + a_0_0_1*x*y - 2*a_0_0_2*beta*z**2 - a_0_0_2*lambda*z**2 + 2*a_0_0_2*x*y*z - a_0_1_0*lambda*y + a_0_1_0*rho*x - a_0_1_0*x*z - a_0_1_0*y - a_0_1_1*beta*y*z - a_0_1_1*lambda*y*z + a_0_1_1*rho*x*z + a_0_1_1*x*y**2 - a_0_1_1*x*z**2 - a_0_1_1*y*z - a_0_2_0*lambda*y**2 + 2*a_0_2_0*rho*x*y - 2*a_0_2_0*x*y*z - 2*a_0_2_0*y**2 - a_1_0_0*lambda*x - a_1_0_0*sigma*x + a_1_0_0*sigma*y - a_1_0_1*beta*x*z - a_1_0_1*lambda*x*z - a_1_0_1*sigma*x*z + a_1_0_1*sigma*y*z + a_1_0_1*x**2*y - a_1_1_0*lambda*x*y + a_1_1_0*rho*x**2 - a_1_1_0*sigma*x*y + a_1_1_0*sigma*y**2 - a_1_1_0*x**2*z - a_1_1_0*x*y - a_2_0_0*lambda*x**2 - 2*a_2_0_0*sigma*x**2 + 2*a_2_0_0*sigma*x*y

## Map monomials to coefficients

Each monomial in the system variables is mapped to the coefficient that must vanish. Solving these coefficient equations is the next step in finding admissible polynomial eigenfunctions.

In [15]:
monomial_coefficients = (
    create_componentwise_expression_with_monomial_bases_from_polynomial_expression(
        remainder, system.get_variables()
    )
)
monomial_coefficients, preliminary_solution = get_reduced_expression(
    monomial_coefficients
)
monomial_coefficients.get_components()

found 15 components
eliminating components with trivial coefficient ...
eliminating 4 components ...
eliminating 1 components ...
no new components to eliminate found. Resuming ...
eliminated 4 coefficients in total


{x**2: -a_2_0_0*lambda - 2*a_2_0_0*sigma,
 x*y*z: 2*a_0_0_2 - 2*a_0_2_0,
 x*y: a_0_0_1 + 2*a_0_2_0*rho + 2*a_2_0_0*sigma,
 x: -a_1_0_0*lambda - a_1_0_0*sigma,
 y**2: -a_0_2_0*lambda - 2*a_0_2_0,
 y: a_1_0_0*sigma,
 z**2: -2*a_0_0_2*beta - a_0_0_2*lambda,
 z: -a_0_0_1*beta - a_0_0_1*lambda,
 1: -a_0_0_0*lambda}

In [17]:
coefficient_equations = list(monomial_coefficients.get_components().values())
groebner_basis = sy.groebner(
    coefficient_equations,
    *monomial_coefficients.get_free_symbols(),
    order="grevlex",
)
groebner_basis

GroebnerBasis([2*a_0_0_1*a_2_0_0*sigma + a_0_0_1**2, 2*a_2_0_0*beta*sigma + 2*a_0_0_1*sigma - 2*a_2_0_0*sigma - a_0_0_1, a_0_0_0*a_2_0_0*sigma, a_0_0_2*a_2_0_0*sigma - a_0_0_2*a_2_0_0, 2*a_2_0_0*sigma**2 + a_0_0_1*sigma - 2*a_2_0_0*sigma - a_0_0_1, a_0_0_1*a_1_0_0, a_0_0_1*beta - 2*a_0_0_1*sigma, a_0_0_0*a_0_0_1, a_0_0_1*lambda + 2*a_0_0_1*sigma, a_1_0_0*lambda, a_0_0_0*lambda, a_2_0_0*lambda + 2*a_2_0_0*sigma, a_0_0_1*a_0_0_2, a_0_0_2*a_1_0_0, a_0_0_2*beta - a_0_0_2, 2*a_0_0_2*rho + 2*a_2_0_0*sigma + a_0_0_1, a_0_0_0*a_0_0_2, a_0_0_2*lambda + 2*a_0_0_2, a_1_0_0*sigma, -a_0_0_2 + a_0_2_0], a_0_0_1, a_1_0_0, beta, rho, a_0_0_0, a_2_0_0, lambda, a_0_2_0, a_0_0_2, sigma, domain='ZZ', order='grevlex')

In [18]:
solutions = sy.solve(
    groebner_basis, monomial_coefficients.get_free_symbols(), dict=True
)
solutions = [
    solution
    for solution in solutions
    if sy.expand(polynomial.sum_up().subs(solution)) != 0
]

In [20]:
for solution in solutions:
    parameters = r",\quad ".join(
        (
            rf"{sy.latex(parameter)} = {sy.latex(solution[parameter])}"
            if parameter in solution
            else sy.latex(parameter)
        )
        for parameter in system.get_parameters()
    )
    eigenvalue = sy.latex(solution.get(lambda_, lambda_))
    eigenfunction = sy.latex(polynomial.sum_up().subs(solution))
    display(
        Math(rf"{parameters}\qquad \lambda = {eigenvalue}\qquad P = {eigenfunction}")
    )

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>